# External data acquisition

**Scientific purpose.** Query and acquire the public HCC-TACE-Seg collection from the
Cancer Imaging Archive/NBIA service for the HCC-only transportability stress test.

**Runnability classification:** runnable from the public collection with network access
and a configured private data root. Downloaded DICOM images, UIDs, and series manifests
remain outside the repository and are not displayed by this notebook.

**Inputs:** public NBIA collection endpoint. **Private outputs:** raw DICOM series,
a resumable series-state file, and a series manifest. Dataset access remains subject to
the source collection terms; this repository does not redistribute images.


In [ ]:
from pathlib import Path
import sys


def locate_repository(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "configs").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from within the cloned repository.")


REPO_ROOT = locate_repository()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from liver_tumor_pipeline.config import load_config, require_path

CONFIG_PATH = REPO_ROOT / "configs" / "paths.yaml"
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(
        "Create an untracked configs/paths.yaml from configs/paths.example.yaml and "
        "set the documented environment variables before running this workflow."
    )

CONFIG = load_config(CONFIG_PATH)
PRIVATE_ARTIFACT_ROOT = require_path(CONFIG, "private_artifact_root", must_exist=False)
OUTPUT_ROOT = require_path(CONFIG, "output_root", must_exist=False)

import json
import re
import shutil
import tempfile
import time
import zipfile

import pandas as pd
import pydicom
import requests

EXTERNAL_ROOT = require_path(CONFIG, "hcc_tace_seg_root", must_exist=False)
ACQUISITION_ROOT = EXTERNAL_ROOT / "acquisition"
RAW_DICOM_ROOT = EXTERNAL_ROOT / "raw_dicom"
SERIES_MANIFEST = ACQUISITION_ROOT / "series_manifest.csv"
STATE_PATH = ACQUISITION_ROOT / "download_state.json"
COLLECTION = "HCC-TACE-Seg"
NBIA_BASE = "https://services.cancerimagingarchive.net/nbia-api/services/v1"
ACQUISITION_ROOT.mkdir(parents=True, exist_ok=True)
RAW_DICOM_ROOT.mkdir(parents=True, exist_ok=True)


## Resumable acquisition helpers

Series UIDs are used only inside private storage. ZIP members are extracted into a
scratch directory and validated as DICOM before being moved to the configured root.


In [ ]:
def safe_series_name(value: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", value)


def read_state() -> dict:
    if not STATE_PATH.is_file():
        return {"collection": COLLECTION, "completed_series": [], "failed_count": 0}
    state = json.loads(STATE_PATH.read_text(encoding="utf-8"))
    if not isinstance(state, dict):
        raise ValueError("Private acquisition state must be a JSON object.")
    state.setdefault("completed_series", [])
    state.setdefault("failed_count", 0)
    return state


def write_state(state: dict) -> None:
    staged_path = STATE_PATH.with_suffix(".tmp")
    staged_path.write_text(json.dumps(state, indent=2), encoding="utf-8")
    staged_path.replace(STATE_PATH)


def valid_dicom_count(root: Path) -> int:
    count = 0
    for path in root.rglob("*"):
        if not path.is_file():
            continue
        try:
            header = pydicom.dcmread(str(path), stop_before_pixels=True, force=True)
            if hasattr(header, "SOPInstanceUID"):
                count += 1
        except Exception:
            continue
    return count


def query_collection_series() -> pd.DataFrame:
    response = requests.get(
        f"{NBIA_BASE}/getSeries",
        params={"Collection": COLLECTION},
        timeout=120,
    )
    response.raise_for_status()
    payload = response.json()
    if not isinstance(payload, list) or not payload:
        raise RuntimeError("NBIA returned no series for HCC-TACE-Seg.")
    table = pd.DataFrame(payload)
    if "SeriesInstanceUID" not in table.columns:
        raise ValueError("NBIA series response lacks SeriesInstanceUID.")
    table.to_csv(SERIES_MANIFEST, index=False)
    return table


def download_one_series(series_uid: str, expected_images: int | None = None) -> int:
    destination = RAW_DICOM_ROOT / safe_series_name(series_uid)
    existing = valid_dicom_count(destination) if destination.is_dir() else 0
    if existing > 0 and (expected_images is None or existing >= expected_images):
        return existing
    scratch_root = Path(tempfile.mkdtemp(prefix="hcc_tace_seg_"))
    try:
        archive = scratch_root / "series.zip"
        response = requests.get(
            f"{NBIA_BASE}/getImage",
            params={"SeriesInstanceUID": series_uid},
            stream=True,
            timeout=120,
        )
        response.raise_for_status()
        with archive.open("wb") as stream:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    stream.write(chunk)
        extracted = scratch_root / "extracted"
        with zipfile.ZipFile(archive) as compressed:
            corrupt_member = compressed.testzip()
            if corrupt_member is not None:
                raise RuntimeError("Downloaded series archive failed ZIP integrity validation.")
            compressed.extractall(extracted)
        count = valid_dicom_count(extracted)
        if count == 0:
            raise RuntimeError("Downloaded series contains no readable DICOM instances.")
        if expected_images is not None and count < expected_images:
            raise RuntimeError("Downloaded series is incomplete relative to NBIA metadata.")
        if destination.exists():
            shutil.rmtree(destination)
        shutil.move(str(extracted), str(destination))
    finally:
        shutil.rmtree(scratch_root, ignore_errors=True)
    return count


def download_collection(series_table: pd.DataFrame, *, retries: int = 5) -> pd.DataFrame:
    state = read_state()
    completed = set(state["completed_series"])
    failed = 0
    for _, series_row in series_table.iterrows():
        series_uid = str(series_row["SeriesInstanceUID"])
        image_count = series_row.get("ImageCount")
        expected_images = int(image_count) if pd.notna(image_count) else None
        if series_uid in completed:
            continue
        last_error = None
        for attempt in range(retries):
            try:
                download_one_series(series_uid, expected_images)
                completed.add(series_uid)
                state["completed_series"] = sorted(completed)
                write_state(state)
                last_error = None
                break
            except Exception as error:
                last_error = type(error).__name__
                if attempt + 1 < retries:
                    time.sleep(20)
        if last_error is not None:
            failed += 1
    state["failed_count"] = failed
    write_state(state)
    return pd.DataFrame(
        {"series_total": [len(series_table)], "series_complete": [len(completed)], "failed": [failed]}
    )


## Deliberate network execution boundary

`query_collection_series()` retrieves and privately stores the current series manifest.
Provide its returned table to `download_collection()` to perform the potentially long
resumable download. Do not move the resulting DICOM files or manifests into this
repository. The reported external analysis ultimately retained 103 known-HCC cases; the
acquisition manifest itself is not a public result.
